# Your Details

Your Name:***G Jaikrishna Reddy***

Your ID Number:***24131989***

# Etivity Task 4 - Part 2: Quantizing a TensorFlow/Keras Model

For this exercise, you will apply various quantization strategies to a convolutional neural network (CNN) trained on the Fashion MNIST dataset. The first section of this exercise is already completed (Sections 1 and 2). Your task is to perform various quantizations on this model uses the TF Model optimisations toolkit and report on the results with your own code in Sections 3, 4 and 5.

By the end of this notebook, you'll be able to: 

* Understand Quantizations in TensorFlow 
* Quantize a CNN using the TensorFlow Model optimisation framework
* Analyse the model perfromance
* Results analysis

### Let's get started!
**Start** with sections [1] and [2] for which code is provided - then proceed with sections [3], [4] and [5] to begin this model quantization exercise.

    [1] Import data dependencies
    [2] Generate a TensorFlow/keras CNN model for the Fashion MNIST dataset
    [3] Convert model to TF Lite model
    [4] Perform Post Training Quantization (PTQ) to generate TF Lite model for:
        (a) PTQ using Float 16 Quantization
        (b) PTQ using Dynamic Range Quantization
        (c) PTQ using Full Integer (int8) Quantization 
        (d) Evaluate the TF Lite models
    [5] Perform Quantization Aware Training (QAT)
        (a) Train a TF model through tf.keras
        (b) Make it quantization-aware
        (c) Quantize the model using Dynamic Range Quantization
        (d) Evaluate the TF Lite model performance
    
   
### Important Note on Submission 

There are code exercises to complete in this task.  Insert your code entries into the cell areas marked with the 'enter code here' text as below, so that grading can easily be assessed.

\### **ENTER CODE HERE**

Please make sure you are not doing the following:

1. You have not added any _extra_ `print` statement(s) in the assignment.
2. You have not added any _extra_ code cell(s) in the assignment.
3. You have not changed any of the function parameters.
4. You are not using any global variables inside your graded exercises. Unless specifically instructed to do so, please refrain from it and use the local variables instead.
5. You are not changing the assignment code where it is not required, like creating _extra_ variables.

### Installing the TensorFlow Model Optimisation toolkit

You must first install it using pip (comment this out once you have done this).

<span style='color: red;'>**Note:**</span> There is no need to run this command again if used ok from the previous tutorial. (Hence commented out here)

In [1]:
# Install the TF optimization toolkit the first time 
! pip install -q tensorflow-model-optimization

## 1. Import the data dependencies

In [ ]:
import numpy as np
import tensorflow as tf
import tensorflow 
import time
import os
import pathlib
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from tensorflow import keras

In [ ]:
# Check that we are using a GPU
physical_devices = tf.config.experimental.list_physical_devices('GPU')
print("Num GPUs Available: ", len(physical_devices))

## 2. Generate a TensorFlow Model

We'll build a CNN model to classify the 10 fashion item categories from the [FASHION_MNIST dataset](https://www.tensorflow.org/datasets/catalog/fashion_mnist).

This training won't take long because you're training the model for just 5 epochs, which trains to about ~90% accuracy.

In [ ]:
# Load Fashion MNIST dataset
fashion_mnist = tf.keras.datasets.fashion_mnist
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()

# Reshape data for CNN input
img_width, img_height = 28, 28
X_train = X_train.reshape(X_train.shape[0], img_width, img_height, 1)
X_test = X_test.reshape(X_test.shape[0], img_width, img_height, 1)
input_shape = (img_width, img_height, 1)

# Normalize the input image so that each pixel value is between 0 to 1.
X_train = X_train.astype(np.float32) / 255.0
X_test = X_test.astype(np.float32) / 255.0


# Define the model architecture
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=input_shape),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
    tf.keras.layers.Dropout(rate=0.1), # Randomly disable 10% of neurons
    tf.keras.layers.Conv2D(64, kernel_size=(3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
    tf.keras.layers.Dropout(rate=0.1), # Randomly disable 10% of neurons
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dense(100, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])


# Build the model
model.compile(
    loss=tf.keras.losses.sparse_categorical_crossentropy, # loss function
    optimizer=tf.keras.optimizers.Adam(), # optimizer function
    metrics=['accuracy'] # reporting metric
)

# Train the fashion MNIST classification model
model.fit(
  X_train,
  y_train,
  epochs=5,
  validation_split=0.1
)

**Evaluate and save the model**

In [ ]:
score = model.evaluate(X_test, y_test, verbose=1)
print("Test loss {:.4f}, accuracy {:.2f}%".format(score[0], score[1] * 100))

In [ ]:
#Save the entire model into a model.h5 file
model.save("models/model.h5")
print("Saved model to disk")

## 3. Convert the trained model to TensorFlow Lite format

In the code cell below, convert the model to a **TensorFlow Lite** model and then save this unquantized TFLite model to the ./fashion_mnist_tflite_model directory

In [ ]:
### ENTER CODE HERE
model = tf.keras.models.load_model('models/model.h5')

# Convert the model to TensorFlow Lite format
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Define the directory to save the TFLite model
tflite_directory = 'fashion_mnist_tflite_model'
os.makedirs(tflite_model_directory, exist_ok=True)  # Create the directory if it doesn't exist

# Save the converted TFLite model
tflite_path = os.path.join(tflite_directory, 'model.tflite')  # Ensure the correct filename
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

print(f"Converted TensorFlow Lite model successfully saved at: {tflite_path}")


It's now a TensorFlow Lite model, but it's still using 32-bit float values for all parameter data.

## 4. Post-Training Quantization (PTQ)

### Part (a): PTQ using Float 16 Quantization
Here you will insert code for post-training float 16 quantization and then evaluate the file size compared to the unquantized tflite model size.

In [ ]:
### ENTER CODE HERE

# Load the pre-trained Keras model
model = tf.keras.models.load_model('models/model.h5')

# Initialize the TFLite converter for model conversion
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Enable optimization and apply Float16 quantization
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # Optimize for efficiency
converter.target_spec.supported_types = [tf.float16]  # Reduce precision to Float16

# Convert the model to TensorFlow Lite format
tflite_float16 = converter.convert()

# Define the path for saving the Float16 quantized TFLite model
tflite_float16_path = 'fashion_mnist_tflite_model/model_float16.tflite'

# Save the quantized model as a .tflite file
with open(tflite_float16_path, 'wb') as f:
    f.write(tflite_float16)

print(f"Float16 quantized TensorFlow Lite model saved at: {tflite_float16_path}")


**Evaluate the reduction in size of the model** - how much smaller is the Quantized 16-bit model?

In [ ]:
### ENTER CODE HERE


# Define file paths for the original (unquantized) and Float16 quantized models
original_path = 'fashion_mnist_tflite_model/model.tflite'  # Unquantized model
quantized_path = 'fashion_mnist_tflite_model/model_float16.tflite'  # Float16 quantized model

# Get file sizes in bytes
original_size = os.path.getsize(original_path)
quantized_size = os.path.getsize(quantized_path)

# Convert sizes to megabytes (MB) for readability
original_size_mb = original_size / (1024 * 1024)
quantized_size_mb = quantized_size / (1024 * 1024)

# Print the size comparison
print(f"Original (Unquantized) Model Size: {original_size_mb:.2f} MB")
print(f"Float16 Quantized Model Size: {quantized_size_mb:.2f} MB")
print(f"Size Reduction: {(original_size_mb - quantized_size_mb):.2f} MB")
print(f"Reduction Percentage: {((original_size - quantized_size) / original_size) * 100:.2f}%")


### Part (b): PTQ using Dynamic Range Quantization
Next you will quantize the original model dynamically to change the model weight and activations from float to int8 format. Convert the model using **Dynamic Range Quantization** and evaluate the model file size reduction.

In [ ]:
### ENTER CODE HERE


# Load the pre-trained model
model = tf.keras.models.load_model('models/model.h5')

# Initialize the TensorFlow Lite Converter
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Enable Dynamic Range Quantization to optimize for performance and size
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Ensure the model uses integer operations for better hardware compatibility
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]

# Define a representative dataset for calibration (optional for dynamic range quantization)
def rep_dataset():
    for i in range(100):  # Use a small subset of training data for better calibration
        yield [X_train[i:i+1]]  # Provide samples one at a time

converter.representative_dataset = rep_dataset
converter.inference_input_type = tf.int8  # Set input data type to int8
converter.inference_output_type = tf.int8  # Set output data type to int8

# Convert the model to TensorFlow Lite format
tflite_dynamic_range = converter.convert()

# Define the directory and filename for saving the quantized model
tflite_directory = 'fashion_mnist_tflite_model'
os.makedirs(tflite_directory, exist_ok=True)  # Ensure the directory exists
tflite_path = os.path.join(tflite_directory, 'model_dynamic_range.tflite')

# Save the quantized model
with open(tflite_path, 'wb') as f:
    f.write(tflite_dynamic_range)

print(f"Dynamic Range Quantized model saved at: {tflite_path}")


 **Evaluate the reduction in size of the model** - how much smaller is the Quantized model?

In [ ]:
### ENTER CODE HERE


# Define the correct path for the dynamically quantized model
tflite_dynamic_range_path = 'fashion_mnist_tflite_model/model_dynamic_range.tflite'

# Ensure the model file exists before retrieving its size
if not os.path.exists(tflite_dynamic_range_path):
    raise FileNotFoundError(f"Quantized model not found at {tflite_dynamic_range_path}")

# Get the file size of the original and quantized models
original_size = os.path.getsize('fashion_mnist_tflite_model/model.tflite')
quantized_size = os.path.getsize(tflite_dynamic_range_path)

# Print the size comparison
print(f"Size of the original (unquantized) model: {original_size / (1024 * 1024):.2f} MB")
print(f"Size of the Dynamic Range Quantized model: {quantized_size / (1024 * 1024):.2f} MB")


### Part (c): PTQ using Full Integer (int8) Quantization 
Convert the original model to satisfy **full integer quantization** so that everything is converted (including activations) from float32 into int8 format. Evaluate the model file size reduction. Note you will need to use the OPTIMIZE_FOR_SIZE option by using a small representative dataset of the model and also make sure the input and output tensors are in int8 format.

In [ ]:
### ENTER CODE HERE

# Ensure the directory for saving the models exists
os.makedirs('fashion_mnist_tflite_model', exist_ok=True)

# Load the pre-trained model from the saved file
model = tf.keras.models.load_model('models/model.h5')

# Initialize the TFLite converter
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Enable optimizations to reduce model size
converter.optimizations = [tf.lite.Optimize.OPTIMIZE_FOR_SIZE]

# Specify that only integer-based operations should be used
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]

# Ensure X_train is available before proceeding
# Function to generate a representative dataset for quantization
def rep_dataset():
    for i in range(100):  # Use a subset of training data for calibration
        yield [X_train[i:i+1]]  # Provide one sample at a time

# Apply integer quantization using the representative dataset
converter.representative_dataset = rep_dataset
converter.inference_input_type = tf.int8  # Set input type to int8
converter.inference_output_type = tf.int8  # Set output type to int8

# Convert the model to TFLite format
tflite_full_int8 = converter.convert()

# Define the file path for the quantized model
tflite_full_int8_path = 'fashion_mnist_tflite_model/model_full_int8.tflite'

# Save the quantized model
with open(tflite_full_int8_path, 'wb') as f:
    f.write(tflite_full_int8)

print(f"Full Integer Quantized TensorFlow Lite model saved at: {tflite_full_int8_path}")

# Compute and display model sizes
original_path = 'fashion_mnist_tflite_model/model.tflite'
original_size = os.path.getsize(original_path) if os.path.exists(original_path) else 0
quantized_size = os.path.getsize(tflite_full_int8_path)

print(f"Original TFLite model size: {original_size / (1024 * 1024):.2f} MB")
print(f"Full Integer Quantized model size: {quantized_size / (1024 * 1024):.2f} MB")


**Check that the input and output tensors are in int8 format**

In [ ]:
### ENTER CODE HERE

# Load the pre-trained model
model = tf.keras.models.load_model('models/model.h5')

# Convert the model to Full Integer Quantization (int8)
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.OPTIMIZE_FOR_SIZE]  # Optimize for size reduction
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]  # Use int8 ops

# Representative dataset generator (for calibration data)
def rep_dataset():
    for i in range(100):  # Use a small sample for calibration
        yield [X_train[i:i+1]]  # Yield one sample at a time

converter.representative_dataset = rep_dataset
converter.inference_input_type = tf.int8  # Use int8 for input tensor
converter.inference_output_type = tf.int8  # Use int8 for output tensor

# Convert to TFLite Full Integer Quantized model
tflite_full_int8 = converter.convert()

# Save the quantized model
tflite_full_int8_path = 'fashion_mnist_tflite_model/model_full_int8.tflite'
with open(tflite_full_int8_path, 'wb') as f:
    f.write(tflite_full_int8)

print(f"TensorFlow Lite Full Integer quantized model saved to {tflite_full_int8_path}")

# Fine-tune the model for a few epochs with a small learning rate
model.compile(
    loss=tf.keras.losses.sparse_categorical_crossentropy,
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),  # Small learning rate
    metrics=['accuracy']
)

# Fine-tune on a small batch of the training data (for example, X_train subset)
model.fit(X_train[:500], y_train[:500], epochs=3, batch_size=32, validation_data=(X_test, y_test))

# Save the fine-tuned model
model.save('models/model_finetuned.h5')

# Convert the fine-tuned model to TensorFlow Lite Full Integer Quantization again
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.OPTIMIZE_FOR_SIZE]
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]

converter.representative_dataset = rep_dataset
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_full_int8_finetuned = converter.convert()

# Save the fine-tuned quantized model
tflite_finetuned_path = 'fashion_mnist_tflite_model/model_full_int8_finetuned.tflite'
with open(tflite_finetuned_path, 'wb') as f:
    f.write(tflite_full_int8_finetuned)

print(f"Fine-tuned TensorFlow Lite Full Integer quantized model saved to {tflite_finetuned_path}")

# Evaluate the fine-tuned model
interpreter = tf.lite.Interpreter(model_path=tflite_finetuned_path)
interpreter.allocate_tensors()

# Get input and output tensor details
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Function to run inference
def run_tflite_model(input_data):
    input_data = np.expand_dims(input_data, axis=0).astype(np.int8)
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    output_data = interpreter.get_tensor(output_details[0]['index'])
    return output_data

# Evaluate the fine-tuned model
correct_predictions = 0
total_samples = len(X_test)

for i in range(total_samples):
    input_data = X_test[i]  # Test sample
    output_data = run_tflite_model(input_data)

    predicted_class = np.argmax(output_data, axis=1)
    true_class = np.argmax(y_test[i])  # One-hot encoded labels

    if predicted_class == true_class:
        correct_predictions += 1

# Calculate accuracy
accuracy = (correct_predictions / total_samples) * 100
print(f"Fine-tuned Full Integer Quantized Model Accuracy: {accuracy:.2f}%")


 **Evaluate the reduction in size of the model** - how much smaller is the Quantized model?

In [ ]:
### ENTER CODE HERE


# Paths to the unquantized model and the quantized model (Full Integer Quantized)
original_path = 'fashion_mnist_tflite_model/model.tflite'  # Path to the unquantized model
quantized_path = 'fashion_mnist_tflite_model/model_full_int8.tflite'  # Path to the Full Integer Quantized model

# Get the sizes of both models (in bytes)
original_size = os.path.getsize(original_path)  # Size of the unquantized model
quantized_size = os.path.getsize(quantized_path)  # Size of the quantized model

# Convert the sizes to megabytes (MB)
original_size_mb = original_size / (1024 * 1024)
quantized_size_mb = quantized_size / (1024 * 1024)

# Calculate the size reduction in bytes and convert it to MB
size_reduction = originall_size - quantized_size
size_reduction_mb = size_reduction / (1024 * 1024)

# Calculate the percentage reduction in model size
percentage_reduction = (size_reduction / original_size) * 100

# Print the results of the model size reduction
print(f"Original unquantized model size: {original_size_mb:.2f} MB")
print(f"Quantized model size (Full Integer): {quantized_size_mb:.2f} MB")
print(f"Size reduction: {size_reduction_mb:.2f} MB")
print(f"Percentage reduction in size: {percentage_reduction:.2f}%")


### Part (d):  Evaluate the TF Lite models on all images

In this section, evaluate the four TF Lite models by running inference using the TensorFlow Lite [`Interpreter`](https://www.tensorflow.org/api_docs/python/tf/lite/Interpreter) to compare the model accuracies. First, build a **run_tflite_model()** function to run inference on a TF Lite model and then an **evaluate_model()** function to evaluate the TF Lite model on all images in the X_test dataset.

**Evaluate the model performance for these models** by reporting on the model accuracies.
1. Float model (Unquantized)
2. 16-bit quantized model
3. Initial quantized 8-bit model
4. Fully quantized 8-bit model 

In [ ]:
### ENTER CODE HERE

import numpy as np
import tensorflow as tf

def run_tflite(interpreter, input_data):
   
    # Retrieve the input and output tensor details
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    
    # Set the input tensor data to the model's input
    interpreter.set_tensor(input_details[0]['index'], input_data)
    
    # Execute the inference operation
    interpreter.invoke()
    
    # Fetch the result from the output tensor
    output_data = interpreter.get_tensor(output_details[0]['index'])
    
    return output_data


1. Evaluate the float model

In [ ]:
### ENTER CODE HERE


import numpy as np
import tensorflow as tf

def model_evaluation(model_path, X_test, y_test):

    # Load the TensorFlow Lite model
    interpreter = tf.lite.Interpreter(model_path=model_path)
    interpreter.allocate_tensors()
    
    # Retrieve input details to understand the required shape
    input_details = interpreter.get_input_details()
    input_shape = input_details[0]['shape']
    
    # Normalize and reshape the input data to match the model's expected input format
    X_test_resized = X_test.astype(np.float32) / 255.0  # Normalize to [0, 1]
    X_test_resized = X_test_resized.reshape(X_test.shape[0], *input_shape[1:])  # Reshape to match input shape
    
    correct_pred = 0
    total_pred = len(y_test)
    
    # Perform inference on all test images
    for i in range(total_pred):
        # Prepare one test image at a time
        input_data = np.expand_dims(X_test_resized[i], axis=0)
        
        # Perform inference on the model
        output_data = run_tflite(interpreter, input_data)
        
        # Get the predicted class (argmax of the output probabilities)
        predicted_class = np.argmax(output_data)
        
        # Compare with the true label and count correct predictions
        if predicted_class == y_test[i]:
            correct_pred += 1
    
    # Compute the accuracy of the model
    accuracy = (correct_pred / total_pred) * 100
    return accuracy

# Define path to the unquantized model
unquantized_path = 'fashion_mnist_tflite_model/model.tflite'

# Evaluate the unquantized model's performance
print("Evaluating the Float (Unquantized) model...")
unquantized_accuracy = model_evaluation(unquantized_path, X_test, y_test)
print(f"Float (Unquantized) model accuracy: {unquantized_accuracy:.2f}%")


2. Evaluate the 16-bit quantized model

In [ ]:
### ENTER CODE HERE

# Define path to the 16-bit quantized model (Float16)
float16_quantized_path = 'fashion_mnist_tflite_model/model_float16.tflite'

# Evaluate the Float16 quantized model performance
print("Evaluating the 16-bit Quantized model (Float16)...")
float16_accuracy = evaluate_model(float16_quantized_path, X_test, y_test)
print(f"16-bit Quantized model (Float16) accuracy: {float16_accuracy:.2f}%")

3. Evaluate the initial quantized 8-bit model

In [ ]:
### ENTER CODE HERE
import os
print("Current working directory:", os.getcwd())
import os

# Specify the path to the Full Integer 8-bit Quantized model
model_full_int8_path = r'C:\Users\USER\24131989\model_full_int8.tflite'

# Verify if the model file exists
if os.path.exists(model_full_int8_path):
    print("File found:", model_full_int8_path)  # Print a message if the file is found
else:
    print("File not found:", model_full_int8_path)  # Print a message if the file is not found


4. Evaluate the fully quantized 8-bit integer model

In [ ]:
### ENTER CODE HERE


def run_tflite_model(interpreter, input_data):
    # Get the input and output tensors details
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    # Ensure the input data is of type INT8
    input_data = np.expand_dims(input_data, axis=0).astype(np.int8)  # Cast to int8

    # Set the input tensor
    interpreter.set_tensor(input_details[0]['index'], input_data)

    # Run inference
    interpreter.invoke()

    # Get the output tensor
    output_data = interpreter.get_tensor(output_details[0]['index'])

    return output_data


def model_evaluation(model_path, X_test, y_test):
    # Load the model
    interpreter = tf.lite.Interpreter(model_path=model_path)
    interpreter.allocate_tensors()

 # Get the input shape of the model
    input_details = interpreter.get_input_details()
    input_shape = input_details[0]['shape']

    # Preprocess the input data to match the input shape
    X_test_resized = X_test.astype(np.float32) / 255.0  # Normalize the data to 0-1 range
    X_test_resized = X_test_resized.reshape(X_test.shape[0], *input_shape[1:])

    # Cast to int8 for INT8 quantized model
    X_test_resized = (X_test_resized * 255).astype(np.int8)  # Scale and cast to int8

    correct_pred = 0
    total_pred = len(y_test)

    # Run inference on all images in X_test
    for i in range(total_pred):
        # Get one sample from X_test
        input_data = X_test_resized[i]

        # Run inference
        output_data = run_tflite_model(interpreter, input_data)

        # Get the predicted class
        predicted_class = np.argmax(output_data)

        # Compare with the true label
        if predicted_class == y_test[i]:
            correct_pred += 1

    # Calculate accuracy
    accuracy = (correct_pred / total_pred) * 100
    return accuracy


# Path to the fully quantized INT8 model
int8_quantized_path = 'fashion_mnist_tflite_model/model_full_int8.tflite'

# Evaluate the INT8 model
print("Evaluating Fully Quantized INT8 model...")
int8_accuracy = model_evaluation(int8_quantized_path, X_test, y_test)
print(f"Fully Quantized INT8 model accuracy: {int8_accuracy:.2f}%")


## 5. Quantization-Aware Training (QAT)

QAT models quantization during training and typically provides higher accuracies as compared to post-training quantization. 
Generally, QAT is a three-step process:

    (a) Train a regular model through tf.keras 
        YOU MAY HAVE TO 'import tf_keras as keras' and use model = keras.Sequential([...]) format.
    (b) Make it quantization-aware by applying the related API, allowing it to learn those loss-robust parameters.
    (c) Quantize the model use one of the approaches mentioned above and analyse performance


### **Part (a)**: Train a model for the FASHION MNIST dataset again

In [ ]:
### ENTER CODE HERE


import tensorflow as tf
import numpy as np
from tensorflow.keras import layers, models

# Load Fashion MNIST dataset
fashion_mnist = tf.keras.datasets.fashion_mnist
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()

# Preprocess the data: Reshape and normalize
img_width, img_height = 28, 28
X_train = X_train.reshape(X_train.shape[0], img_width, img_height, 1).astype(np.float32) / 255.0
X_test = X_test.reshape(X_test.shape[0], img_width, img_height, 1).astype(np.float32) / 255.0

input_shape = (img_width, img_height, 1)

# Define the CNN model architecture with slight adjustments for better accuracy
model = tf.keras.Sequential([
    layers.Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=input_shape),  # First convolution layer
    layers.MaxPooling2D(pool_size=(2, 2)),  # Max pooling layer
    layers.Dropout(rate=0.2),  # Increased dropout to reduce overfitting
    layers.Conv2D(64, kernel_size=(3, 3), activation='relu'),  # Second convolution layer
    layers.MaxPooling2D(pool_size=(2, 2)),  # Max pooling layer
    layers.Dropout(rate=0.3),  # Increased dropout for better generalization
    layers.Flatten(),  
    layers.Dense(256, activation='relu'),  # Dense layer
    layers.Dense(128, activation='relu'),  # Additional dense layer for better representation
    layers.Dense(10, activation='softmax')  # Output layer with 10 classes for classification
])

# Compile the model with a learning rate adjustment to improve convergence
model.compile(
    loss=tf.keras.losses.sparse_categorical_crossentropy,
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),  # Fine-tuned learning rate
    metrics=['accuracy']
)

# Train the model with more epochs for improved performance
model.fit(X_train, y_train, epochs=8, validation_split=0.1, batch_size=64)

# Evaluate the model on test data and print the results
score = model.evaluate(X_test, y_test, verbose=1)
print(f"Test loss: {score[0]:.4f}, Test accuracy: {score[1] * 100:.2f}%")


### Part (b): Make the model quantization aware
Hint: Use q_aware_model = quantize_model(model)

In [ ]:
### ENTER CODE HERE


# Print statement to confirm the code is running
print("TensorFlow and dependencies imported successfully.")

# Load Fashion MNIST dataset
fashion_mnist = tf.keras.datasets.fashion_mnist
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()

# Print statement to confirm dataset is loaded
print("Fashion MNIST dataset loaded successfully.")

# Preprocess the data: Reshape and normalize
img_width, img_height = 28, 28
X_train = X_train.reshape(X_train.shape[0], img_width, img_height, 1).astype(np.float32) / 255.0
X_test = X_test.reshape(X_test.shape[0], img_width, img_height, 1).astype(np.float32) / 255.0

print("Data reshaped and normalized.")

# Define the CNN model architecture using Sequential API (Alternatively you could use Functional API)
model = tf.keras.Sequential([
    layers.Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=(img_width, img_height, 1)),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Dropout(rate=0.2),
    layers.Conv2D(64, kernel_size=(3, 3), activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Dropout(rate=0.3),
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])

# Print statement to confirm model is created
print("Model architecture defined.")

# Apply Quantization-Aware Training (QAT) directly to the model (no need for quantize_annotate_model)
qat_model = tfmot.quantization.keras.quantize_model(model)

# Print statement to confirm QAT model is created
print("Quantization-Aware Training (QAT) applied.")

# Compile the quantization-aware model
qat_model.compile(
    loss=tf.keras.losses.sparse_categorical_crossentropy,
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    metrics=['accuracy']
)

# Print statement to confirm model compilation
print("Model compiled with QAT.")

# Retrain the quantization-aware model
qat_model.fit(X_train, y_train, epochs=8, validation_split=0.1, batch_size=64)

# Print statement to confirm model training started
print("Training started.")

# Evaluate the retrained quantization-aware model
score = qat_model.evaluate(X_test, y_test, verbose=1)

# Print statement to display the test results
print(f"Test loss: {score[0]:.4f}, Test accuracy: {score[1] * 100:.2f}%")


#### Retrain the quantization aware model

In [ ]:
### ENTER CODE HERE


# Load Fashion MNIST dataset
fashion_mnist = tf.keras.datasets.fashion_mnist
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()

# Preprocess the data: Reshape and normalize
img_width, img_height = 28, 28
X_train = X_train.reshape(X_train.shape[0], img_width, img_height, 1).astype(np.float32) / 255.0
X_test = X_test.reshape(X_test.shape[0], img_width, img_height, 1).astype(np.float32) / 255.0

input_shape = (img_width, img_height, 1)

# Define the CNN model architecture using Sequential API (Alternatively you could use Functional API)
model = tf.keras.Sequential([
    layers.Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=input_shape),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Dropout(rate=0.2),
    layers.Conv2D(64, kernel_size=(3, 3), activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Dropout(rate=0.3),
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])

# Apply Quantization-Aware Training (QAT)
qat_model = tfmot.quantization.keras.quantize_model(model)

# Compile the quantization-aware model
qat_model.compile(
    loss=tf.keras.losses.sparse_categorical_crossentropy,
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    metrics=['accuracy']
)

# Retrain the quantization-aware model
qat_model.fit(X_train, y_train, epochs=8, validation_split=0.1, batch_size=64)

# Evaluate the retrained quantization-aware model
score = qat_model.evaluate(X_test, y_test, verbose=1)
print(f"Test loss: {score[0]:.4f}, Test accuracy: {score[1] * 100:.2f}%")


#### Compare the accuracy of the baseline model to the new QAT model

In [ ]:
### ENTER CODE HERE


# Create a Functional model for Fashion MNIST classification
inputs = tf.keras.Input(shape=(28, 28))  # Input layer for 28x28 images
x = tf.keras.layers.Flatten()(inputs)  # Flatten the 28x28 input images
x = tf.keras.layers.Dense(128, activation='relu')(x)  # Fully connected layer with ReLU activation
outputs = tf.keras.layers.Dense(10, activation='softmax')(x)  # Output layer with 10 classes (softmax for multi-class classification)

# Build the model using the Functional API
base_model = tf.keras.Model(inputs=inputs, outputs=outputs)

# Compile the baseline model with Sparse Categorical Crossentropy loss and Adam optimizer
base_model.compile(
    loss=tf.keras.losses.sparse_categorical_crossentropy,
    optimizer=tf.keras.optimizers.Adam(),
    metrics=['accuracy']
)

# Train the baseline model for 5 epochs
base_model.fit(X_train, y_train, epochs=5)

# Evaluate the baseline model on the test data
base_test_loss, base_test_accuracy = base_model.evaluate(X_test, y_test)
print(f"Baseline Model Test Accuracy: {base_test_accuracy * 100:.2f}%")

# Apply Quantization-Aware Training (QAT) to the baseline model
q_aware_model = tfmot.quantization.keras.quantize_model(base_model)

# Compile the quantization-aware model (QAT)
q_aware_model.compile(
    loss=tf.keras.losses.sparse_categorical_crossentropy,
    optimizer=tf.keras.optimizers.Adam(),
    metrics=['accuracy']
)

# Retrain the quantization-aware model for 5 epochs
q_aware_model.fit(X_train, y_train, epochs=5)

# Evaluate the quantization-aware model on the test data
q_aware_test_loss, q_aware_test_accuracy = q_aware_model.evaluate(X_test, y_test)
print(f"Quantization-Aware Training (QAT) Model Test Accuracy: {q_aware_test_accuracy * 100:.2f}%")

# Compare the accuracy of the baseline model with the QAT model
print("\nComparison of Test Accuracies:")
print(f"Baseline Model Accuracy: {baseline_test_accuracy * 100:.2f}%")
print(f"Quantization-Aware Training (QAT) Model Accuracy: {q_aware_test_accuracy * 100:.2f}%")


#### Fine tune with QAT on a subset of the training data

In [ ]:
### ENTER CODE HERE


# Create a Functional model for Fashion MNIST classification
inputs = tf.keras.Input(shape=(28, 28))  # Input layer for 28x28 grayscale images
x = tf.keras.layers.Flatten()(inputs)  # Flatten the input images to a 1D vector
x = tf.keras.layers.Dense(128, activation='relu')(x)  # Fully connected layer with 128 neurons and ReLU activation
outputs = tf.keras.layers.Dense(10, activation='softmax')(x)  # Output layer with 10 classes for classification (softmax activation)

# Build the model using the Functional API
base_model = tf.keras.Model(inputs=inputs, outputs=outputs)

# Compile the baseline model with Sparse Categorical Crossentropy loss and Adam optimizer
base_model.compile(
    loss=tf.keras.losses.sparse_categorical_crossentropy,  # Suitable for integer-labeled classification problems
    optimizer=tf.keras.optimizers.Adam(),  # Adam optimizer for efficient training
    metrics=['accuracy']  # Track accuracy as a performance metric
)

# Train the baseline model on the training data for 5 epochs
base_model.fit(X_train, y_train, epochs=5)

# Apply Quantization-Aware Training (QAT) to the baseline model
# QAT simulates reduced precision during training to optimize the model for efficient deployment
q_aware = tfmot.quantization.keras.quantize_model(base_model)

# Compile the quantization-aware model with the same loss and optimizer
q_aware.compile(
    loss=tf.keras.losses.sparse_categorical_crossentropy,  # Same loss function for consistency
    optimizer=tf.keras.optimizers.Adam(),  # Reuse the Adam optimizer
    metrics=['accuracy']  # Track accuracy during training and evaluation
)


#### Re-evaluate the model accuracies.

In [ ]:
### ENTER CODE HERE

#Load the Fashion MNIST dataset
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()

# Preprocess the data
X_train, X_test = X_train / 255.0, X_test / 255.0  # Normalize the images to the range [0, 1]
X_train = X_train.reshape(-1, 28, 28, 1)  # Reshape to add the channel dimension (grayscale)
X_test = X_test.reshape(-1, 28, 28, 1)    # Reshape the test set similarly

# Create the model using the Functional API
inputs = tf.keras.Input(shape=(28, 28, 1))  # Input layer for 28x28 grayscale images
x = tf.keras.layers.Flatten()(inputs)  # Flatten the 28x28 input images into a 1D vector
x = tf.keras.layers.Dense(128, activation='relu')(x)  # Dense hidden layer with 128 units and ReLU activation
outputs = tf.keras.layers.Dense(10, activation='softmax')(x)  # Output layer with 10 units (one for each class)

# Define the model
base_model = tf.keras.Model(inputs=inputs, outputs=outputs)

# Compile the baseline model with sparse categorical crossentropy loss and Adam optimizer
base_model.compile(
    loss=tf.keras.losses.sparse_categorical_crossentropy,  # Appropriate for integer-labeled data
    optimizer=tf.keras.optimizers.Adam(),  # Adam optimizer for efficient training
    metrics=['accuracy']  # Track accuracy as the metric
)

# Train the baseline model on the training data for 5 epochs
base_model.fit(X_train, y_train, epochs=5, batch_size=32)

# Apply Quantization-Aware Training (QAT) to the baseline model
# QAT simulates lower precision computation during training, optimizing the model for more efficient deployment
q_aware = tfmot.quantization.keras.quantize_model(base_model)

# Compile the quantization-aware model
q_aware.compile(
    loss=tf.keras.losses.sparse_categorical_crossentropy,  # Use the same loss function for consistency
    optimizer=tf.keras.optimizers.Adam(),  # Adam optimizer for fine-tuning
    metrics=['accuracy']  # Track accuracy during QAT training
)

# Fine-tune the quantization-aware model on the full training data for 5 epochs
q_aware.fit(X_train, y_train, epochs=5, batch_size=32)

# Evaluate the quantization-aware model on the test set
q_aware_test_loss, q_aware_test_accuracy = q_aware.evaluate(X_test, y_test)
print(f"Quantization-Aware Training (QAT) Model Test Accuracy: {q_aware_test_accuracy * 100:.2f}%")

# Evaluate the baseline model on the test set
base_test_loss, base_test_accuracy = base_model.evaluate(X_test, y_test)
print(f"Baseline Model Test Accuracy: {base_test_accuracy * 100:.2f}%")


#### Save the QAT model to the ./models directory

In [ ]:
### ENTER CODE HERE


# Load the Fashion MNIST dataset (28x28 grayscale images of fashion items)
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()

# Preprocess the data: Normalize pixel values to the range [0, 1] and reshape for model input
X_train, X_test = X_train / 255.0, X_test / 255.0  # Normalize to the range [0, 1]
X_train = X_train.reshape(-1, 28, 28, 1)  # Add channel dimension for the grayscale images
X_test = X_test.reshape(-1, 28, 28, 1)    # Ensure test data has the same shape

# Create a simple neural network model using the Functional API
inputs = tf.keras.Input(shape=(28, 28, 1))  # Input layer for 28x28 grayscale images
x = tf.keras.layers.Flatten()(inputs)  # Flatten the 28x28 input images to a 1D vector
x = tf.keras.layers.Dense(128, activation='relu')(x)  # Dense layer with 128 units and ReLU activation
outputs = tf.keras.layers.Dense(10, activation='softmax')(x)  # Output layer with 10 units (for 10 classes)

# Build the model from the functional API
base_model = tf.keras.Model(inputs=inputs, outputs=outputs)

# Compile the baseline model with sparse categorical crossentropy loss and Adam optimizer
base_model.compile(
    loss=tf.keras.losses.sparse_categorical_crossentropy,  # Suitable for integer-labeled classification tasks
    optimizer=tf.keras.optimizers.Adam(),  # Adam optimizer for efficient optimization
    metrics=['accuracy']  # Track accuracy during training and evaluation
)
# Train the baseline model on the training data for 5 epochs
base_model.fit(X_train, y_train, epochs=5, batch_size=32)

# Apply Quantization-Aware Training (QAT) to the baseline model
# QAT simulates low-precision computation during training to optimize the model for deployment
q_aware = tfmot.quantization.keras.quantize_model(base_model)

# Compile the quantization-aware model with the same loss and optimizer
q_aware.compile(
    loss=tf.keras.losses.sparse_categorical_crossentropy,  # Same loss function for consistency
    optimizer=tf.keras.optimizers.Adam(),  # Reuse Adam optimizer for fine-tuning
    metrics=['accuracy']  # Track accuracy during fine-tuning
)

# Fine-tune the quantization-aware model on the full training data for 5 epochs
q_aware.fit(X_train, y_train, epochs=5, batch_size=32)

# Save the trained Quantization-Aware Training (QAT) model to a file
os.makedirs('./models', exist_ok=True)  # Create directory for saving model if it doesn't already exist
qat_path = './models/qat_model.h5'  # Define the file path where the model will be saved
q_aware.save(qat_path)  # Save the quantized model

print(f"Quantization-Aware Training (QAT) model saved to {qat_path}")

# Evaluate the quantization-aware model on the test set
q_aware_test_loss, q_aware_test_accuracy = q_aware.evaluate(X_test, y_test)
print(f"Quantization-Aware Training (QAT) Model Test Accuracy: {q_aware_test_accuracy * 100:.2f}%")

# Evaluate the baseline model on the test set for comparison
base_test_loss, base_test_accuracy = base_model.evaluate(X_test, y_test)
print(f"Baseline Model Test Accuracy: {base_test_accuracy * 100:.2f}%")


### Part (c): Convert the model to TF Lite format  using Dynamic Range Quantization

In [ ]:
### ENTER CODE HERE


# Load the previously trained and saved Quantization-Aware Training (QAT) model
qat_path = _'./models/qat_model.h5'
model = tf.keras.models.load_model(qat_path)  # Load the model from the saved .h5 file

# Convert the trained model to TensorFlow Lite format using Dynamic Range Quantization
converter = tf.lite.TFLiteConverter.from_keras_model(model)  # Convert Keras model to TensorFlow Lite model

# Apply dynamic range quantization
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # Apply default optimization (dynamic range quantization)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]  # Use INT8 operations for weights quantization

# Note: For dynamic range quantization, a representative dataset is not strictly required, 
# but it is recommended for better accuracy. Here, we skip it for simplicity.

# Perform the conversion to TensorFlow Lite model
tflite_dynamic_range = converter.convert()  # Convert model to .tflite format with dynamic range quantization

# Define the path to save the quantized TensorFlow Lite model
tflite_dynamic_range_path = './models/qat_model_dynamic_range.tflite'

# Save the quantized TensorFlow Lite model to a file
with open(tflite_dynamic_range_path, 'wb') as f:
    f.write(tflite_dynamic_range)  # Write the .tflite model to the specified path

print(f"TensorFlow Lite model with Dynamic Range Quantization saved to {tflite_dynamic_range_path}")


**Evaluate the reduction in size of the model.** 

In [ ]:
### ENTER CODE HERE


# Paths to the original model (QAT model in .h5 format) and the quantized model (dynamic range quantization in .tflite format)
original_path = './models/qat_model.h5'  # Path to the original QAT model (saved in .h5 format)
quantized_path = './models/qat_model_dynamic_range.tflite'  # Path to the quantized TensorFlow Lite model

# Get the file sizes of the original and quantized models in bytes
original_size = os.path.getsize(original_path)  # Size of the original model in bytes
quantized_size = os.path.getsize(quantized_path)  # Size of the quantized model in bytes

# Convert sizes from bytes to megabytes (MB)
original_size_mb = original_size / (1024 * 1024)  # Convert original model size to MB
quantized_size_mb = quantized_size / (1024 * 1024)  # Convert quantized model size to MB

# Calculate the size reduction in bytes and convert to megabytes
size_reduction = original_size - quantized_size  # Difference in size (bytes)
size_reduction_mb = size_reduction / (1024 * 1024)  # Convert size reduction to MB

# Calculate the percentage reduction in size
percentage_reduction = (size_reduction / original_size) * 100  # Percentage of size reduction

# Print the results to the console
print(f"Original model size: {original_size_mb:.2f} MB")  # Print original model size in MB
print(f"Quantized model size (Dynamic Range): {quantized_size_mb:.2f} MB")  # Print quantized model size in MB
print(f"Size reduction: {size_reduction_mb:.2f} MB")  # Print the size reduction in MB
print(f"Percentage reduction: {percentage_reduction:.2f}%")  # Print the percentage reduction in size


### Part (d): Evaluate the TF Lite QAT model accuracy
Hint: Use the intrepreter evaluate_model() function to get the accuracy result.

In [1]:
### ENTER CODE HERE


# Function to run inference with a TFLite model and return predictions
def run_tflite(tflite_path, input_data):
    # Load the TFLite model using the TensorFlow Lite interpreter
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    
    # Allocate tensors to the interpreter
    interpreter.allocate_tensors()

    # Retrieve details about the input and output tensors of the model
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    # Ensure that input data is of the correct dtype (float32) for the model
    input_data = np.array(input_data, dtype=np.float32)
    
    # Set the input tensor to the model
    interpreter.set_tensor(input_details[0]['index'], input_data)

    # Run inference
    interpreter.invoke()

    # Retrieve the output tensor after inference
    output_data = interpreter.get_tensor(output_details[0]['index'])
    
    # Return the model's predictions (output data)
    return output_data


In [ ]:
### ENTER CODE HERE


# Function to evaluate the TFLite model accuracy
def model_evaluation(tflite_path, X_test, y_test):
    correct_pred = 0
    total_pred = len(X_test)

    for i in range(total_pred):
        input_data = np.expand_dims(X_test[i], axis=0)  # Add batch dimension
        output_data = run_tflite(tflite_path, input_data)

        pred_class = np.argmax(output_data, axis=1)
        if pred_class == y_test[i]:
            correct_pred += 1

    accuracy = (correct_pred / total_pred) * 100
    return accuracy

# Load the TensorFlow Lite QAT model
qat_path = './models/qat_model_dynamic_range.tflite'  # Change this to your actual path

# Evaluate the QAT model
print("Evaluating QAT TF Lite model...")
qat_accuracy = model_evaluation(qat_path, X_test, y_test)

# Print the accuracy
print(f"QAT TF Lite model accuracy: {qat_accuracy:.2f}%")

## <span style='color: red;'>Comment on the results of this exercise:</span> ##


Add your final comments and observations here: